<a href="https://colab.research.google.com/github/TrollRider-Kristian/Springboard-AI-Mini-Projects/blob/main/Student_MLE_MiniProject_Recommendation_Engines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Recommendation Engines

Recommendation engines are algorithms designed to provide personalized suggestions or recommendations to users. These systems analyze user behavior, preferences, and interactions with items (products, movies, music, articles, etc.) to predict and offer items that users are likely to be interested in. Recommendation engines play a crucial role in enhancing user experience, driving engagement, and increasing conversion rates in various applications, including e-commerce, entertainment, content platforms, and more.

There are generally two approaches taken in collaborative filtering and content-based recommendation engines:

**1. Collaborative Filtering:**
Collaborative Filtering is a popular approach to building recommendation systems that leverages the collective behavior of users to make personalized recommendations. It is based on the idea that users who have agreed in the past will likely agree in the future. There are two main types of collaborative filtering:

- **User-based Collaborative Filtering:** This method finds users similar to the target user based on their past interactions (e.g., ratings or purchases). It then recommends items that similar users have liked but the target user has not interacted with yet.

- **Item-based Collaborative Filtering:** In this approach, the system identifies similar items based on user interactions. It recommends items that are similar to the ones the target user has already liked or interacted with.

Collaborative filtering does not require any explicit information about items but relies on the similarity between users or items. It is effective in capturing complex patterns and can provide serendipitous recommendations. However, it suffers from the cold-start problem (i.e., difficulty in recommending to new users or items with no interactions) and scalability challenges in large datasets.

**2. Content-Based Recommendation:**
Content-based recommendation is an alternative approach to building recommendation systems that focuses on the attributes or features of items and users. It leverages the characteristics of items to make recommendations. The key steps involved in content-based recommendation are:

- **Feature Extraction:** For each item, relevant features are extracted. For movies, these features could be genre, director, actors, and plot summary.

- **User Profile:** A user profile is created based on the items they have interacted with in the past. The user profile contains the weighted importance of features based on their interactions.

- **Similarity Calculation:** The similarity between items or between items and the user profile is calculated using similarity metrics like cosine similarity or Euclidean distance.

- **Recommendation:** Items that are most similar to the user profile are recommended to the user.

Content-based recommendation systems are less affected by the cold-start problem as they can still recommend items based on their features. They are also more interpretable as they rely on item attributes. However, they may miss out on providing serendipitous recommendations and can be limited by the quality of feature extraction and user profiles.

**Choosing Between Collaborative Filtering and Content-Based:**
Both collaborative filtering and content-based approaches have their strengths and weaknesses. The choice between them depends on the specific requirements of the recommendation system, the type of data available, and the user base. Hybrid approaches that combine collaborative filtering and content-based techniques are also common, aiming to leverage the strengths of both methods and mitigate their weaknesses.

In this mini-project, you'll be building both content based and collaborative filtering engines for the [MovieLens 25M dataset](https://grouplens.org/datasets/movielens/25m/). The MovieLens 25M dataset is one of the most widely used and popular datasets for building and evaluating recommendation systems. It is provided by the GroupLens Research project, which collects and studies datasets related to movie ratings and recommendations. The MovieLens 25M dataset contains movie ratings and other related information contributed by users of the MovieLens website.

**Dataset Details:**
- **Size:** The dataset contains approximately 25 million movie ratings.
- **Users:** It includes ratings from over 162,000 users.
- **Movies:** The dataset consists of ratings for more than 62,000 movies.
- **Ratings:** The ratings are provided on a scale of 1 to 5, where 1 is the lowest rating and 5 is the highest.
- **Timestamps:** Each rating is associated with a timestamp, indicating when the rating was given.

**Data Files:**
The dataset is usually split into three CSV files:

1. **movies.csv:** Contains information about movies, including the movie ID, title, genres, and release year.
   - Columns: movieId, title, genres

2. **ratings.csv:** Contains movie ratings provided by users, including the user ID, movie ID, rating, and timestamp.
   - Columns: userId, movieId, rating, timestamp

3. **tags.csv:** Contains user-generated tags for movies, including the user ID, movie ID, tag, and timestamp.
   - Columns: userId, movieId, tag, timestamp

First, import all the libraries you'll need.

In [12]:
import zipfile
import numpy as np
import pandas as pd
from urllib.request import urlretrieve
from sklearn.metrics.pairwise import cosine_similarity

Next, download the relevant components of the MoveLens dataset. Note, these instructions are roughly based on the colab [here](https://colab.research.google.com/github/google/eng-edu/blob/main/ml/recommendation-systems/recommendation-systems.ipynb?utm_source=ss-recommendation-systems&utm_campaign=colab-external&utm_medium=referral&utm_content=recommendation-systems#scrollTo=O3bcgduFo4s6).

In [13]:
print("Downloading movielens data...")

urlretrieve('http://files.grouplens.org/datasets/movielens/ml-100k.zip', 'movielens.zip')
zip_ref = zipfile.ZipFile('movielens.zip', 'r')
zip_ref.extractall()
print("Done. Dataset contains:")
print(zip_ref.read('ml-100k/u.info'))

ratings_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t', names=ratings_cols, encoding='latin-1')

# The movies file contains a binary feature for each genre.
genre_cols = [
    "genre_unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
movies_cols = [
    'movie_id', 'title', 'release_date', "video_release_date", "imdb_url"
] + genre_cols
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', names=movies_cols, encoding='latin-1')

Done. Dataset contains:
b'943 users\n1682 items\n100000 ratings\n'


Before doing any kind of machine learning, it's always good to familiarize yourself with the datasets you'lll be working with.

Here are your tasks:

1. Spend some time familiarizing yourself with both the `movies` and `ratings` dataframes. How many unique user ids are present? How many unique movies are there?
2. Create a new dataframe that merges the `movies` and `ratings` tables on 'movie_id'. Only keep the 'user_id', 'title', 'rating' fields in this new dataframe.

In [14]:
# KRISTIAN_NOTE - Start with a preview of the elements in the movies and ratings dataframes
print("MOVIES DATASET -------")
print(movies.head())
print("RATINGS DATASET -------")
print(ratings.head())

MOVIES DATASET -------
   movie_id              title release_date  video_release_date  \
0         1   Toy Story (1995)  01-Jan-1995                 NaN   
1         2   GoldenEye (1995)  01-Jan-1995                 NaN   
2         3  Four Rooms (1995)  01-Jan-1995                 NaN   
3         4  Get Shorty (1995)  01-Jan-1995                 NaN   
4         5     Copycat (1995)  01-Jan-1995                 NaN   

                                            imdb_url  genre_unknown  Action  \
0  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
1  http://us.imdb.com/M/title-exact?GoldenEye%20(...              0       1   
2  http://us.imdb.com/M/title-exact?Four%20Rooms%...              0       0   
3  http://us.imdb.com/M/title-exact?Get%20Shorty%...              0       1   
4  http://us.imdb.com/M/title-exact?Copycat%20(1995)              0       0   

   Adventure  Animation  Children  ...  Fantasy  Film-Noir  Horror  Musical  \
0          0        

In [5]:
# KRISTIAN_NOTE - I noticed some one-hot encoding for the genre of each film.
# I wonder what all the genres are.  The first five columns are metadata around
# movie_id, title, release_date, video_release_date, and imdb_url.
print(movies.columns[5:])
print("There are " + str(len(movies.columns[5:])) + " distinct genres in the movies dataframe.")

Index(['genre_unknown', 'Action', 'Adventure', 'Animation', 'Children',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')
There are 19 distinct genres in the movies dataframe.


Indices 5 through 23 are genre-specific and one-hot encoded for the movies dataframe.  That means a dataframe consisting of genres alone would be 19 columns long.  This will become relevant later in this project.

In [15]:
# Spend some time familiarizing yourself with both the movies and ratings
# dataframes. How many unique user ids are present? How many unique movies
# are there?
print("The movies dataframe is of size: " + str(movies.shape))
# KRISTIAN_NOTE - Fortunately, Pandas 3 came out with a field that
# counts distinct entries along a column.  See the docs for pd.unique here:
# https://pandas.pydata.org/docs/reference/api/pandas.unique.html#pandas.unique
print("The number of unique movies in the movies dataframe is: " + str(len(pd.unique(movies['movie_id']))))
print("The ratings dataframe is of size: " + str(ratings.shape))
print("The number of unique users in the ratings dataframe is: " + str(len(pd.unique(ratings['user_id']))))
print("The number of unique movies in the ratings dataframe is: " + str(len(pd.unique(ratings['movie_id']))))
print("The number of movie_id values present in both movies and ratings is: " +\
      str(movies['movie_id'].isin(ratings['movie_id']).count()))

The movies dataframe is of size: (1682, 24)
The number of unique movies in the movies dataframe is: 1682
The ratings dataframe is of size: (100000, 4)
The number of unique users in the ratings dataframe is: 943
The number of unique movies in the ratings dataframe is: 1682
The number of movie_id values present in both movies and ratings is: 1682


Although there are 100,000 user reviews in the ratings dataframe, the number of unique movies in both dataframes is the same: 1682.  Since movies are identified by movie_id, and they both possess the same 1682 movie_id values, that means they share the exact same movies, and the merge function is good enough to piece together the two dataframes by movie_id.

In [16]:
# Merge movies and ratings
movies_and_ratings = pd.merge(movies, ratings, on = "movie_id")
# KRISTIAN_NOTE - Yes, the same movie is going to repeat a bunch of times, but
# the user_id and rating is different for each row.  Need a different row for
# each user who reviews the movie because we want it all in the same dataframe.
print(movies_and_ratings.head())
print("The size of the merged dataframe is: " + str(movies_and_ratings.shape))

   movie_id             title release_date  video_release_date  \
0         1  Toy Story (1995)  01-Jan-1995                 NaN   
1         1  Toy Story (1995)  01-Jan-1995                 NaN   
2         1  Toy Story (1995)  01-Jan-1995                 NaN   
3         1  Toy Story (1995)  01-Jan-1995                 NaN   
4         1  Toy Story (1995)  01-Jan-1995                 NaN   

                                            imdb_url  genre_unknown  Action  \
0  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
1  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
2  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
3  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
4  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   

   Adventure  Animation  Children  ...  Musical  Mystery  Romance  Sci-Fi  \
0          0          1         1  ...        0    

The fact that there are an equal number of rows in the merged dataframe as in the ratings dataframe suggests that the merge function works as intended.

As mentioned in the introduction, content-Based Filtering is a recommendation engine approach that focuses on the attributes or features of items (products, movies, music, articles, etc.) and leverages these features to make personalized recommendations. The underlying idea is to match the characteristics of items with the preferences of users to suggest items that align with their interests. Content-based filtering is particularly useful when explicit user-item interactions (e.g., ratings or purchases) are sparse or unavailable.

**Key Steps in Content-Based Filtering:**

1. **Feature Extraction:**
   - For each item, relevant features are extracted. These features are typically descriptive attributes that can be represented numerically, such as genre, director, actors, author, publication date, and keywords.
   - In the case of text-based items, natural language processing techniques may be used to extract features like TF-IDF (Term Frequency-Inverse Document Frequency) scores.

2. **User Profile Creation:**
   - A user profile is created based on the items they have interacted with in the past. The user profile contains the weighted importance of features based on their interactions.
   - For example, if a user has watched several action movies, the action genre feature would receive a higher weight in their profile.

3. **Similarity Calculation:**
   - The similarity between items or between items and the user profile is calculated using similarity metrics like cosine similarity, Euclidean distance, or Pearson correlation.
   - Cosine similarity is commonly used as it measures the cosine of the angle between two vectors, which represents their similarity.

4. **Recommendation:**
   - Items that are most similar to the user profile are recommended to the user. These are items whose features have the highest similarity scores with the user profile.
   - The recommended items are presented as a list sorted by their similarity scores.

**Advantages of Content-Based Filtering:**
1. **No Cold-Start Problem:** Content-based filtering can make recommendations even for new users with no historical interactions because it relies on item features rather than user history.

2. **User Independence:** The recommendations are based solely on the features of items and do not require knowledge of other users' preferences or behavior.

3. **Transparency:** Content-based recommendations are interpretable, as they depend on the features of items, making it easier for users to understand why specific items are recommended.

4. **Serendipity:** Content-based filtering can recommend items with characteristics not seen before by the user, leading to serendipitous discoveries.

5. **Diversity in Recommendations:** The method can offer diverse recommendations since it suggests items with different feature combinations.

**Limitations of Content-Based Filtering:**
1. **Limited Discovery:** Content-based filtering may struggle to recommend items outside the scope of users' historical interactions or interests.

2. **Over-Specialization:** Users may receive recommendations that are too similar to their previous choices, leading to a lack of exposure to new item categories.

3. **Dependency on Feature Quality:** The quality and relevance of item features significantly influence the quality of recommendations.

4. **Limited for Cold Items:** Content-based filtering can struggle to recommend new items with limited feature information.

Here is your task:

1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return content-based recommendations for this user. Here are steps you can take:

  A. Get the user's rated movies

  B. Create a TF-IDF matrix using movie genres. Note, this can be extracted from the `movies` dataframe.

  C. Compute the cosine similarity between movie genres. Use the [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) function.

  D. Get the indices of similar movies to those rated by the user based on cosine similarity. Keep only the top 5.

  E. Remove duplicates and movies already rated by the user.

In [59]:
# STEP_1 - Get the user's rated movies from most to least loved
def get_rated_movies_by_user(user_id, df):
  return df.loc[df['user_id'] == user_id, :].sort_values(by=['rating'], ascending = False)

test_user_308 = get_rated_movies_by_user(308, movies_and_ratings)
print(test_user_308.loc[:, ['user_id', 'title', 'rating']].head())
test_user_247 = get_rated_movies_by_user(247, movies_and_ratings)
print(test_user_247.loc[:, ['user_id', 'title', 'rating']].head())
test_user_148 = get_rated_movies_by_user(148, movies_and_ratings)
print(test_user_148.loc[:, ['user_id', 'title', 'rating']].head())

       user_id                      title  rating
67154      308     Wings of Desire (1987)       5
66971      308  Lawrence of Arabia (1962)       5
66033      308             Bananas (1971)       5
65689      308  African Queen, The (1951)       5
65271      308     His Girl Friday (1940)       5
       user_id                                title  rating
6453       247                     Star Wars (1977)       5
4651       247                     Apollo 13 (1995)       5
15466      247  Truth About Cats & Dogs, The (1996)       5
9877       247   Four Weddings and a Funeral (1994)       5
8700       247     Shawshank Redemption, The (1994)       5
       user_id                           title  rating
1053       148           Twelve Monkeys (1995)       5
24841      148  Raiders of the Lost Ark (1981)       5
6535       148                Star Wars (1977)       5
7719       148             Pulp Fiction (1994)       5
9427       148             Forrest Gump (1994)       5


In [60]:
# STEP 2 - Create a TF-IDF matrix using movie genres.
# The kicker is that since the movie genres are already one-hot encoded,
# we don't need a TF-IDF vectorizer because the movies are already "vectorized".
def get_genres_only_matrix (df, genres_starting_index):
  genres_list = list (df.columns[genres_starting_index:])
  genres_only_df = df.loc[:, genres_list]
  genres_only_df.index = df['title']
  return genres_only_df

genres_only_df = get_genres_only_matrix(movies, 5)
print(genres_only_df.head())
print("And the size of the genres only dataframe is: " + str(genres_only_df.shape))

                   genre_unknown  Action  Adventure  Animation  Children  \
title                                                                      
Toy Story (1995)               0       0          0          1         1   
GoldenEye (1995)               0       1          1          0         0   
Four Rooms (1995)              0       0          0          0         0   
Get Shorty (1995)              0       1          0          0         0   
Copycat (1995)                 0       0          0          0         0   

                   Comedy  Crime  Documentary  Drama  Fantasy  Film-Noir  \
title                                                                      
Toy Story (1995)        1      0            0      0        0          0   
GoldenEye (1995)        0      0            0      0        0          0   
Four Rooms (1995)       0      0            0      0        0          0   
Get Shorty (1995)       1      0            0      1        0          0   
Copycat (19

Since we already have a vector of 1's and 0's for the genres of each movie title, cosine similarity will work on the movies matrix as-is without the need for a TF-IDF vectorizer.  For example, "Goldeneye" and "Copycat" share 1 genre in common (Thriller) and receive a similarity score of 0.33333.

In [61]:
# STEP_3 - Get the indices of the similar movies based on cosine similarity
# Do this on the genres-only dataframe we extracted in STEP_2.
def get_cosine_similarity_df (df, category):
  return pd.DataFrame(cosine_similarity(df), index=category, columns=category)

movie_similarity_df = get_cosine_similarity_df(genres_only_df, genres_only_df.index) # index = title
print(movie_similarity_df.head())
print(movie_similarity_df.shape)

title              Toy Story (1995)  GoldenEye (1995)  Four Rooms (1995)  \
title                                                                      
Toy Story (1995)           1.000000          0.000000            0.00000   
GoldenEye (1995)           0.000000          1.000000            0.57735   
Four Rooms (1995)          0.000000          0.577350            1.00000   
Get Shorty (1995)          0.333333          0.333333            0.00000   
Copycat (1995)             0.000000          0.333333            0.57735   

title              Get Shorty (1995)  Copycat (1995)  \
title                                                  
Toy Story (1995)            0.333333        0.000000   
GoldenEye (1995)            0.333333        0.333333   
Four Rooms (1995)           0.000000        0.577350   
Get Shorty (1995)           1.000000        0.333333   
Copycat (1995)              0.333333        1.000000   

title              Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)  \

In [62]:
# STEP 4 -
def get_cosine_similarities_to_rated_movies_by_user (user_id):
  rated_titles = list(get_rated_movies_by_user(user_id, movies_and_ratings)['title'])
  print(rated_titles)
  cosine_similarities_of_rated_titles = movie_similarity_df.loc[movie_similarity_df.index.isin(rated_titles)]
  print(cosine_similarities_of_rated_titles.head())

test_rated_movies_user_308 = get_cosine_similarities_to_rated_movies_by_user(308)

['Wings of Desire (1987)', 'Lawrence of Arabia (1962)', 'Bananas (1971)', 'African Queen, The (1951)', 'His Girl Friday (1940)', 'Pulp Fiction (1994)', 'Some Like It Hot (1959)', 'Vertigo (1958)', 'Big Sleep, The (1946)', 'Close Shave, A (1995)', 'Short Cuts (1993)', 'Red Rock West (1992)', 'Heathers (1989)', 'Harold and Maude (1971)', 'E.T. the Extra-Terrestrial (1982)', 'Beauty and the Beast (1991)', 'Speed (1994)', 'Blade Runner (1982)', 'Hudsucker Proxy, The (1994)', 'Braindead (1992)', 'Face/Off (1997)', 'Secrets & Lies (1996)', 'Rear Window (1954)', 'Raising Arizona (1987)', 'Chinatown (1974)', 'Touch of Evil (1958)', 'Once Upon a Time in the West (1969)', 'Grifters, The (1990)', 'Paris, Texas (1984)', 'Being There (1979)', 'Cape Fear (1991)', 'Unforgiven (1992)', 'Evil Dead II (1987)', 'Shining, The (1980)', 'Citizen Kane (1941)', 'Bound (1996)', 'Fargo (1996)', 'Platoon (1986)', 'Godfather: Part II, The (1974)', 'GoodFellas (1990)', "Singin' in the Rain (1952)", 'Good, The Bad 

In [ ]:
# Content-Based Filtering using Movie Genres
def content_based_recommendation(user_id, df):
  # Get the user's rated movies

  # Create a TF-IDF matrix using movie genres

  # Compute the cosine similarity between movie genres

  # Get the indices of the similar movies to those rated by the user based on cosine similarity

  # Remove duplicates and movies already rated by the user

The key idea behind collaborative filtering is that users who have agreed in the past will likely agree in the future. Instead of relying on item attributes or user profiles, collaborative filtering identifies patterns of user behavior and item preferences from the interactions present in the data.

**Types of Collaborative Filtering:**
There are two main types of collaborative filtering:

**Collaborative Filtering Process:**
The collaborative filtering process typically involves the following steps:

1. **Data Collection:**
   - Gather data on user-item interactions, such as movie ratings, product purchases, or article clicks.

2. **User-Item Matrix:**
   - Organize the data into a user-item matrix, where rows represent users, columns represent items, and the entries contain the users' interactions (e.g., ratings).

3. **Similarity Calculation:**
   - Calculate the similarity between users or items using similarity metrics such as cosine similarity, Pearson correlation, or Jaccard similarity.
   - For user-based collaborative filtering, user similarities are calculated, and for item-based collaborative filtering, item similarities are calculated.

4. **Neighborhood Selection:**
   - For each user or item, select the most similar users or items as the neighborhood.
   - The size of the neighborhood (the number of similar users or items to consider) is an important parameter to control the system's behavior.

5. **Prediction Generation:**
   - Predict the ratings for items that the target user has not yet interacted with by combining the ratings of neighboring users or items.

6. **Recommendation Generation:**
   - Recommend items with the highest predicted ratings to the target user.

**Advantages of Collaborative Filtering using User-Item Interactions:**
- Collaborative filtering is based solely on user interactions and does not require knowledge of item attributes, making it useful for cases where item data is sparse or unavailable.
- It can provide serendipitous recommendations, suggesting items that users may not have discovered on their own.
- Collaborative filtering can be applied in various domains, including e-commerce, music, movie, and content recommendations.

**Limitations of Collaborative Filtering:**
- The cold-start problem: Collaborative filtering struggles to recommend to new users or items with no or limited interaction history.
- It may suffer from sparsity when data is limited or when users have only interacted with a small subset of items.
- Scalability issues can arise with large datasets and an increasing number of users or items.

Here is your task:

1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return collaborative filtering recommendations for this user based on a user-item interaction matrix. Here are steps you can take:

  A. Create the user-item matrix using Pandas' [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html).

  B. Fill missing values with zeros in this matrix.

  C. Calculate user-user similarity matrix using cosine similarity.

  D. Get the array of similarity scores of the target user with all other users from the similarity matrix.

  E. Extract, say the the top 5 most similar users (excluding the target user).

  F. Generate movie recommendations based on the most similar users.

  G. Remove duplicate movies recommendations.

In [ ]:
# KRISTIAN_NOTE - The pivot_table function also accepts a fill value as an extra parameter,
# so I can fill all the missing ratings with 0 in the same function call.
def get_users_and_titles_df (df):
  return pd.pivot_table(df, values='rating', index='user_id', columns='title', fill_value=0)

users_and_titles_df = get_users_and_titles_df(movies_and_ratings)
print(users_and_titles_df.head())

title    'Til There Was You (1997)  1-900 (1994)  101 Dalmatians (1996)  \
user_id                                                                   
1                              0.0           0.0                    2.0   
2                              0.0           0.0                    0.0   
3                              0.0           0.0                    0.0   
4                              0.0           0.0                    0.0   
5                              0.0           0.0                    2.0   

title    12 Angry Men (1957)  187 (1997)  2 Days in the Valley (1996)  \
user_id                                                                 
1                        5.0         0.0                          0.0   
2                        0.0         0.0                          0.0   
3                        0.0         2.0                          0.0   
4                        0.0         0.0                          0.0   
5                        0.0        

In [ ]:
# Collaborative Filtering using User-Item Interactions
def collaborative_filtering_recommendation(user_id, df):
  # Create the user-item matrix

  # Fill missing values with 0 (indicating no rating)

  # Calculate user-user similarity matrix using cosine similarity

  # Get the similarity scores of the target user with all other users

  # Find the top N most similar users (excluding the target user)

  # Generate movie recommendations based on the most similar users

  # Remove duplicates from recommendations

Now, test your recommendations engines! Select a few user ids and generate recommendations using both functions you've written. Are the recommendations similar? Do the recommendations make sense?

In [ ]:
# Test the recommendation engines